# TUTORIAL 05: OMNI-DIMENSIONAL HOMEOMORPHISMS (UPDATED)

Different dimensions use different classification machinery. This notebook shows the current `pysurgery` behavior across 2D, 3D, 4D, and n>=5, including decision-ready certificate paths introduced in recent phases.

In [1]:
import numpy as np
from scipy.sparse import csr_matrix

from pysurgery import (
    ChainComplex,
    FundamentalGroup,
    IntersectionForm,
    ObstructionResult,
    WhiteheadGroup,
    analyze_homeomorphism_2d,
    analyze_homeomorphism_3d,
    analyze_homeomorphism_4d,
    analyze_homeomorphism_high_dim,
    ThreeManifoldRecognitionCertificate,
    HomotopyCompletionCertificate,
    ProductAssemblyCertificate,
    DefiniteLatticeIsometryCertificate,
)

## 1) 2D: closed surfaces

In 2D, orientability and low-dimensional homology determine the classification branch in this API.

In [2]:
c2a = ChainComplex(
    boundaries={1: csr_matrix((1, 0)), 2: csr_matrix((0, 1))}, dimensions=[0, 1, 2]
)
c2b = ChainComplex(
    boundaries={1: csr_matrix((1, 0)), 2: csr_matrix((0, 1))}, dimensions=[0, 1, 2]
)
ok2, msg2 = analyze_homeomorphism_2d(c2a, c2b)
print(ok2, msg2)

True SUCCESS: Homeomorphism established via the Classification Theorem of Closed Surfaces using exact H_1/H_2 invariants.


## 2) 3D: recognition certificate completion

`analyze_homeomorphism_3d` can remain inconclusive on homology/fundamental-group evidence alone, but now supports decision-ready `recognition_certificate` inputs for certified completion in supported branches.

In [3]:
d1 = csr_matrix(np.zeros((1, 1), dtype=np.int64))
d2 = csr_matrix(np.array([[3]], dtype=np.int64))
d3 = csr_matrix(np.zeros((1, 1), dtype=np.int64))
c3 = ChainComplex(
    boundaries={1: d1, 2: d2, 3: d3},
    dimensions=[0, 1, 2, 3],
    cells={0: 1, 1: 1, 2: 1, 3: 1},
)
pi = FundamentalGroup(generators=["g"], relations=[["g", "g", "g"]])

ok3_plain, msg3_plain = analyze_homeomorphism_3d(c3, c3)
print("without recognition_certificate:", ok3_plain, msg3_plain[:110])

rec = ThreeManifoldRecognitionCertificate(
    provided=True, source="tutorial", exact=True, validated=True
)
ok3_cert, msg3_cert = analyze_homeomorphism_3d(c3, c3, recognition_certificate=rec)
print("with recognition_certificate:", ok3_cert, msg3_cert[:110])

without recognition_certificate: None INCONCLUSIVE: Manifolds are homology equivalent. In 3D, full homeomorphism recognition requires geometric/fund
with recognition_certificate: True SUCCESS: Homology/fundamental-group compatibility checks pass and a decision-ready 3-manifold recognition cert


## 3) 4D: Freedman branch + definite certificate path

Definite forms may require explicit isometry data. The API accepts a decision-ready `definite_lattice_isometry_certificate`.

In [4]:
q1 = np.array([[1, 0], [0, 1]], dtype=np.int64)
u = np.array([[3, 2], [2, 1]], dtype=np.int64)
q2 = u.T @ q1 @ u
f1 = IntersectionForm(matrix=q1, dimension=4)
f2 = IntersectionForm(matrix=q2, dimension=4)

cert4 = DefiniteLatticeIsometryCertificate(
    provided=True,
    source="tutorial",
    exact=True,
    validated=True,
    isometry_matrix=u.tolist(),
)
ok4, msg4 = analyze_homeomorphism_4d(
    f1,
    f2,
    ks1=0,
    ks2=0,
    simply_connected=True,
    definite_lattice_isometry_certificate=cert4,
)
print(ok4, msg4[:110])

True SUCCESS: Definite lattice isomorphism certificate found (U^T Q1 U = Q2).


## 4) n>=5: obstruction/certificate-driven classification

High-dimensional success is certificate-gated: Whitehead + Wall + completion evidence, and for product groups, assembly evidence.

In [5]:
d1h = csr_matrix(np.zeros((1, 0), dtype=np.int64))
d5h = csr_matrix(np.zeros((0, 1), dtype=np.int64))
c5 = ChainComplex(
    boundaries={1: d1h, 5: d5h},
    dimensions=[0, 1, 2, 3, 4, 5],
    cells={0: 1, 1: 0, 2: 0, 3: 0, 4: 0, 5: 1},
)
wh = WhiteheadGroup(rank=0, description="Wh(Z_2 x Z_3)=0", computable=True, exact=True)
wall = ObstructionResult(
    dimension=5,
    pi="Z_2 x Z_3",
    computable=True,
    exact=True,
    value=0,
    modulus=None,
    assumptions=[],
    decomposition_kind="factor_surrogate",
    assembly_certified=False,
    obstructs=False,
    zero_certified=True,
)

completion = HomotopyCompletionCertificate(
    provided=True, source="tutorial", exact=True, validated=True
)
assembly = ProductAssemblyCertificate(
    provided=True, source="tutorial", exact=True, validated=True
)

ok5, msg5 = analyze_homeomorphism_high_dim(
    c5,
    c5,
    dim=5,
    pi_group="Z_2 x Z_3",
    whitehead_group=wh,
    wall_obstruction=wall,
    homotopy_completion_certificate=completion,
    product_assembly_certificate=assembly,
)
print(ok5, msg5[:120])

True SUCCESS: Homology equivalence, Wh(pi_1)=0, vanishing Wall obstruction, and an exact validated homotopy-completion certif


For explicit witness maps, obstruction interference examples, and surgery planning statements, see `examples/08_certificate_workflows.ipynb`.